# Remove semantically duplicate images
> Remove duplicate images based on embedding from pretrained net

In [ ]:
#| default_exp preprocessing.semantic_deduplication

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| export
import sys
from pathlib import Path
from fastcore.script import *
from fastcore.all import *
import os


In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))


In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))


In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from dotenv import load_dotenv


In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintercra/homes/hasan/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintercra/homes/hasan/projects/git_data/new_test/nbs'

In [ ]:
#| export
from PIL import Image
import open_clip
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import ResNet50_Weights
import torchvision.models as models
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
import pickle
import hashlib
from typing import List, Tuple, Dict, Set
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import faiss
import argparse


In [ ]:
#| export
class SemanticImageDeduplicator:
    """
    Advanced semantic image deduplication system using deep learning features.
    Supports multiple similarity metrics and clustering algorithms.
    """
    
    def __init__(self, 
                 model_name='resnet50', # CNN model for feature extraction ('resnet50', 'efficientnet', 'vit')
                 similarity_threshold=0.85, # Cosine similarity threshold for duplicates (0.0-1.0)
                 use_gpu=True, # Whether to use GPU acceleration
                 batch_size=32, # Batch size for feature extraction
                 num_workers=4 # Number of worker threads
                 ):
        """
        Initialize the semantic deduplicator.
        
        """
        self.similarity_threshold = similarity_threshold
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
        
        # Initialize feature extractor
        self.feature_extractor = self._init_feature_extractor(model_name)
        self.feature_dim = self._get_feature_dim(model_name)
        
        # Image preprocessing
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        # Storage for features and metadata
        self.image_features = {}
        self.image_paths = []
        self.duplicate_groups = []
        
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
    
    def _init_feature_extractor(
        self, 
        model_name: str
        ) -> nn.Module:
        """Initialize the CNN feature extractor."""
        if model_name == 'resnet50':
            model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
            model = nn.Sequential(*list(model.children())[:-1])  # Remove classifier
        elif model_name == 'efficientnet':
            model = models.efficientnet_b0(pretrained=True)
            model.classifier = nn.Identity()  # Remove classifier
        elif model_name == 'vit':
            model = models.vit_b_16(pretrained=True)
            model.heads = nn.Identity()  # Remove classification head
        else:
            raise ValueError(f"Unsupported model: {model_name}")
        
        model.eval()
        model.to(self.device)
        return model
    
    def _get_feature_dim(
        self, 
        model_name: str # CNN model for feature extraction ('resnet50', 'efficientnet', 'vit')
        ) -> int:
        """Get feature dimension for the model."""
        dims = {
            'resnet50': 2048,
            'efficientnet': 1280,
            'vit': 768
        }
        return dims.get(model_name, 2048)
   

In [ ]:
#| export
@patch
def _load_and_preprocess_image(
    self:SemanticImageDeduplicator,  
    image_path: str
    ) -> torch.Tensor:
    """Load and preprocess a single image."""
    try:
        image = Image.open(image_path).convert('RGB')
        return self.transform(image)
    except Exception as e:
        self.logger.error(f"Error loading image {image_path}: {e}")
        return None


In [ ]:
#| export
@patch
def _extract_features_batch(
    self:SemanticImageDeduplicator, 
    image_batch: torch.Tensor
    ) -> np.ndarray:
    """Extract features from a batch of images."""
    with torch.no_grad():
        features = self.feature_extractor(image_batch.to(self.device))
        if len(features.shape) > 2:
            features = torch.flatten(features, start_dim=1)
        return features.cpu().numpy()

In [ ]:
#| export
@patch
def extract_features(
    self:SemanticImageDeduplicator, 
    image_paths: List[str],  # List of image file paths
    save_cache: str = None # Path to save feature cache
    ) -> Dict[str, np.ndarray]: # Dictionary mapping image paths to feature vectors
    """
    Extract deep learning features from images.
    """
    self.logger.info(f"Extracting features from {len(image_paths)} images...")
        
    features_dict = {}
    valid_paths = []
        
    # Load images in batches
    for i in tqdm(range(0, len(image_paths), self.batch_size)):
        batch_paths = image_paths[i:i + self.batch_size]
        batch_images = []
        batch_valid_paths = []
            
        for path in batch_paths:
            img_tensor = self._load_and_preprocess_image(path)
            if img_tensor is not None:
                batch_images.append(img_tensor)
                batch_valid_paths.append(path)
            
        if batch_images:
            batch_tensor = torch.stack(batch_images)
            batch_features = self._extract_features_batch(batch_tensor)
                
            for path, features in zip(batch_valid_paths, batch_features):
                # L2 normalize features for better cosine similarity
                features = features / (np.linalg.norm(features) + 1e-8)
                features_dict[path] = features
                valid_paths.append(path)
        
    self.image_features = features_dict
    self.image_paths = valid_paths
        
    # Cache features if requested
    if save_cache:
        with open(save_cache, 'wb') as f:
            pickle.dump(features_dict, f)
        self.logger.info(f"Features cached to {save_cache}")
        
    return features_dict

In [ ]:
#| export
@patch
def load_features_cache(
    self:SemanticImageDeduplicator, 
    cache_path: str # Path to cache file
    ) -> Dict[str, np.ndarray]: # Dictionary mapping image paths to feature vectors
    """Load pre-computed features from cache."""
    with open(cache_path, 'rb') as f:
        self.image_features = pickle.load(f)
    self.image_paths = list(self.image_features.keys())
    self.logger.info(f"Loaded {len(self.image_features)} cached features")
    return self.image_features

In [ ]:
#| export
@patch
def compute_similarity_matrix(
    self:SemanticImageDeduplicator
    ) -> np.ndarray: # Cosine similarity matrix between all image features
    if not self.image_features:
        raise ValueError("No features extracted. Call extract_features() first.")
        
    self.logger.info("Computing similarity matrix...")
    feature_matrix = np.stack(list(self.image_features.values()))
    similarity_matrix = cosine_similarity(feature_matrix)
        
    return similarity_matrix
    

In [ ]:
#| export
@patch
def find_duplicates_threshold(self:SemanticImageDeduplicator) -> List[List[str]]:
    """Find duplicate groups using similarity threshold."""
    similarity_matrix = self.compute_similarity_matrix()
        
    duplicate_groups = []
    processed = set()
        
    for i, path_i in enumerate(self.image_paths):
        if path_i in processed:
            continue
            
        # Find all images similar to current image
        similar_indices = np.where(similarity_matrix[i] >= self.similarity_threshold)[0]
            
        if len(similar_indices) > 1:  # More than just itself
            group = [self.image_paths[idx] for idx in similar_indices]
            duplicate_groups.append(group)
            processed.update(group)
        
    self.duplicate_groups = duplicate_groups
    return duplicate_groups

In [ ]:
#| export
@patch
def find_duplicates_clustering(
    self:SemanticImageDeduplicator, 
    eps:float=0.15, # Maximum distance between samples in the same cluster
    min_samples:int=2 # Minimum number of samples in a cluster
    ) -> List[List[str]]: # List of duplicate groups
    """Find duplicate groups using DBSCAN clustering."""
    if not self.image_features:
        raise ValueError("No features extracted. Call extract_features() first.")
        
        self.logger.info("Finding duplicates using DBSCAN clustering...")
        
        # Convert similarity to distance for DBSCAN
        feature_matrix = np.stack(list(self.image_features.values()))
        
        # Use DBSCAN with cosine distance
        clustering = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine')
        cluster_labels = clustering.fit_predict(feature_matrix)
        
        # Group images by cluster
        clusters = {}
        for idx, label in enumerate(cluster_labels):
            if label != -1:  # -1 means noise/singleton
                if label not in clusters:
                    clusters[label] = []
                clusters[label].append(self.image_paths[idx])
        
        # Filter out single-image clusters
        duplicate_groups = [group for group in clusters.values() if len(group) > 1]
        
        self.duplicate_groups = duplicate_groups
        return duplicate_groups

In [ ]:
#| export
@patch
def find_duplicates_faiss(
    self:SemanticImageDeduplicator, 
    k:int=10 # Number of nearest neighbors to consider
    ) -> List[List[str]]: # List of duplicate groups
    """Find duplicates using FAISS for efficient similarity search."""
    if not self.image_features:
        raise ValueError("No features extracted. Call extract_features() first.")
        
    self.logger.info("Finding duplicates using FAISS...")
        
    # Build FAISS index
    feature_matrix = np.stack(list(self.image_features.values())).astype('float32')
        
    # Use inner product search (equivalent to cosine similarity for normalized vectors)
    index = faiss.IndexFlatIP(self.feature_dim)
    index.add(feature_matrix)
        
    # Search for similar images
    similarities, indices = index.search(feature_matrix, k)
        
    duplicate_groups = []
    processed = set()
        
    for i, (sim_scores, sim_indices) in enumerate(zip(similarities, indices)):
        if i in processed:
            continue
            
        # Find indices with similarity above threshold
        valid_indices = sim_indices[sim_scores >= self.similarity_threshold]
            
        if len(valid_indices) > 1:
            group = [self.image_paths[idx] for idx in valid_indices]
            duplicate_groups.append(group)
            processed.update(valid_indices)
        
    self.duplicate_groups = duplicate_groups
    return duplicate_groups

In [ ]:
#| export
@patch
def select_best_images(
    self:SemanticImageDeduplicator, 
    strategy:str='largest_file' # Selection strategy ('largest_file', 'highest_resolution', 'random')
    ) -> Dict[int, str]: # Dictionary mapping group index to selected image path
    """
    Select the best representative image from each duplicate group.
        
    Args:
        strategy: Selection strategy ('largest_file', 'highest_resolution', 'random')
            
    Returns:
        Dictionary mapping group index to selected image path
    """
    selected_images = {}
        
    for group_idx, group in enumerate(self.duplicate_groups):
        if strategy == 'largest_file':
            # Select image with largest file size
            best_image = max(group, key=lambda x: os.path.getsize(x))
        elif strategy == 'highest_resolution':
            # Select image with highest resolution
            def get_resolution(path):
                try:
                    with Image.open(path) as img:
                        return img.width * img.height
                except:
                    return 0
            best_image = max(group, key=get_resolution)
        elif strategy == 'random':
            # Select random image (actually first one for consistency)
            best_image = group[0]
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
            
        selected_images[group_idx] = best_image
        
    return selected_images

In [ ]:
#| export
@patch
def remove_duplicates(
    self:SemanticImageDeduplicator, 
    output_dir: str = None, # Directory to move kept images (if None, delete duplicates in place)
    dry_run: bool = True, # If True, only report what would be done without actual deletion
    selection_strategy: str = 'largest_file' # Strategy for selecting best image from each group
    remove_duplicates: bool = True # If True, remove duplicates
    ) -> Dict: # Statistics about the deduplication process
    """
    Remove duplicate images, keeping only the best representative.
    """
    if not self.duplicate_groups:
        raise ValueError("No duplicate groups found. Call find_duplicates_*() first.")
        
    selected_images = self.select_best_images(selection_strategy)
        
    stats = {
        'total_images': len(self.image_paths),
        'duplicate_groups': len(self.duplicate_groups),
        'total_duplicates': sum(len(group) for group in self.duplicate_groups),
        'images_to_remove': 0,
        'images_to_keep': len(selected_images),
        'space_saved': 0
    }
        
    images_to_remove = []
        
    for group_idx, group in enumerate(self.duplicate_groups):
        kept_image = selected_images[group_idx]
            
        for image_path in group:
            if image_path != kept_image:
                images_to_remove.append(image_path)
                if os.path.exists(image_path):
                    stats['space_saved'] += os.path.getsize(image_path)
        
    stats['images_to_remove'] = len(images_to_remove)
        
    if dry_run:
        self.logger.info(f"DRY RUN: Would remove {len(images_to_remove)} duplicate images")
        self.logger.info(f"Space that would be saved: {stats['space_saved'] / (1024*1024):.2f} MB")
    else:
        # Actually remove files
        if output_dir:
            Path(output_dir).mkdir(parents=True, exist_ok=True)
            # Move kept images to output directory
            for kept_image in selected_images.values():
                if Path(kept_image).exists():
                    dest_path = Path(output_dir) / Path(kept_image).name
                    Path(kept_image).rename(dest_path)
            
        # Remove duplicate images
        if remove_duplicates:
            for image_path in images_to_remove:
                if Path(image_path).exists():
                    Path(image_path).unlink()
                    self.logger.info(f"Removed: {image_path}")
        
    return stats

In [ ]:
#| export
@patch
def generate_report(
    self:SemanticImageDeduplicator, 
    output_file: str = "deduplication_report.txt" # Path to save the report
    ):
    """Generate a detailed deduplication report."""
    if not self.duplicate_groups:
        self.logger.warning("No duplicate groups to report.")
        return
        
    with open(output_file, 'w') as f:
        f.write("SEMANTIC IMAGE DEDUPLICATION REPORT\n")
        f.write("=" * 50 + "\n\n")
            
        f.write(f"Total images processed: {len(self.image_paths)}\n")
        f.write(f"Duplicate groups found: {len(self.duplicate_groups)}\n")
        f.write(f"Similarity threshold: {self.similarity_threshold}\n\n")
            
        for i, group in enumerate(self.duplicate_groups):
            f.write(f"Duplicate Group {i+1} ({len(group)} images):\n")
            for j, image_path in enumerate(group):
                size = os.path.getsize(image_path) if os.path.exists(image_path) else 0
                f.write(f"  {j+1}. {image_path} ({size} bytes)\n")
            f.write("\n")
        
    self.logger.info(f"Report saved to {output_file}")


In [ ]:
#| export   
def get_image_files(
    directory: str, # Directory to search for images
    extensions: Set[str] = None # Set of file extensions to consider
    ) -> List[str]: # List of image file paths
    """Recursively get all image files from directory."""
    if extensions is None:
        extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    
    image_files = []
    for root, dirs, files in tqdm(os.walk(directory), desc="Scanning directories"):
        for file in files:
            if Path(file).suffix.lower() in extensions:
                image_files.append(os.path.join(root, file))
    
    return image_files


In [ ]:
#| export
class MultiModalFeatureExtractor:
    """
    Multi-modal feature extractor combining DINOv2, CLIP, and ResNet50
    for robust semantic image representation
    """
    
    def __init__(
        self, 
        device='cuda' if torch.cuda.is_available() else 'cpu'
        ):
        # Setup logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        self.device = device
        self.logger.info(f"Using device: {self.device}")
        
        # Initialize models
        self._load_dinov2()
        self._load_clip()
        self._load_resnet()
        
        # Image preprocessing
        self.dinov2_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        self.clip_preprocess = None  # Will be set when CLIP load

In [ ]:
#| export
@patch
def _load_dinov2(
    self:SemanticImageDeduplicator
    ):
        try:
            # Load DINOv2 ViT-Base model
            self.dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
            self.dinov2_model.to(self.device)
            self.dinov2_model.eval()
            self.logger.info("DINOv2 model loaded successfully")
        except Exception as e:
            self.logger.error(f"Failed to load DINOv2: {e}")
            self.dinov2_model = None

In [ ]:
#| export
def _load_clip(
    self:MultiModalFeatureExtractor
    ):
        """Load CLIP model"""
        try:
            self.clip_model, self.clip_preprocess = clip.load("ViT-B/32", device=self.device)
            self.clip_model.eval()
            self.logger.info("CLIP model loaded successfully")
        except Exception as e:
            self.logger.error(f"Failed to load CLIP: {e}")
            self.clip_model = None

In [ ]:
#| export
def _load_resnet(
    self:MultiModalFeatureExtractor
    ):
    """Load ResNet50 model"""
    try:
        model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        model = nn.Sequential(*list(model.children())[:-1])  # Remove classifier
        model.eval()
        model.to(self.device)
        self.logger.info("ResNet50 model loaded successfully")
    except Exception as e:
        self.logger.error(f"Failed to load ResNet50: {e}")
        self.resnet_model = None

In [ ]:
#| export
@patch
def extract_dinov2_features(
    self:MultiModalFeatureExtractor, 
    image: Image.Image
    ) -> np.ndarray:
        """Extract features using DINOv2"""
        if self.dinov2_model is None:
            print("DINOv2 model not loaded, load the model first")
            return np.array([])
        
        with torch.no_grad():
            img_tensor = self.dinov2_transform(image).unsqueeze(0).to(self.device)
            features = self.dinov2_model(img_tensor)
            return features.cpu().numpy().flatten()

In [ ]:
#| export
@patch
def extract_clip_features(
    self:MultiModalFeatureExtractor, 
    image: Image.Image
    ) -> np.ndarray:
        """Extract features using CLIP"""
        if self.clip_model is None:
            print("CLIP model not loaded, load the model first")
            return np.array([])
        
        with torch.no_grad():
            img_tensor = self.clip_preprocess(image).unsqueeze(0).to(self.device)
            features = self.clip_model.encode_image(img_tensor)
            return features.cpu().numpy().flatten()

In [ ]:
#| export
@patch
def extract_resnet_features(
    self:MultiModalFeatureExtractor, 
    image: Image.Image
    ) -> np.ndarray:
        """Extract features using ResNet50"""
        if self.resnet_model is None:
            print("ResNet50 model not loaded, load the model first")
            return np.array([])
        
        with torch.no_grad():
            img_tensor = self.resnet_transform(image).unsqueeze(0).to(self.device)
            features = self.resnet_model(img_tensor)
            return features.cpu().numpy().flatten()

In [ ]:
#| export
def extract_combined_features(
    self:MultiModalFeatureExtractor, 
    image: Image.Image
    ) -> np.ndarray:
        """Extract combined features using DINOv2, CLIP, and ResNet50"""
        if weights is None:
            weights = [0.5, 0.3, 0.2]
        if self.dinov2_model is None or self.clip_model is None or self.resnet_model is None:
            print("Models are not loaded, load the models first")
            return np.array([])
        
        with torch.no_grad():
            pass

In [ ]:
#| export
def parse_args_():
    parser = argparse.ArgumentParser(description='Semantic Image Deduplication')
    parser.add_argument('--input-dir', help='Input directory containing images')
    parser.add_argument('--output-dir', help='Output directory for kept images')
    parser.add_argument('--model', default='resnet50', choices=['resnet50', 'efficientnet', 'vit'],
                       help='Feature extraction model')
    parser.add_argument('--threshold', type=float, default=0.85,
                       help='Similarity threshold for duplicates')
    parser.add_argument('--method', default='threshold', choices=['threshold', 'clustering', 'faiss'],
                       help='Duplicate detection method')
    parser.add_argument('--cache', help='Path to save/load feature cache')
    parser.add_argument('--dry-run', action='store_true', help='Show what would be done without actually doing it')
    parser.add_argument('--remove-duplicates', action='store_true', help='Remove duplicates')
    parser.add_argument('--strategy', default='largest_file', choices=['largest_file', 'highest_resolution', 'random'],
                       help='Strategy for selecting best image from duplicates')
    return parser.parse_args()

In [ ]:
#| export
def main():
    """Main function for command-line usage."""
    args = parse_args_() 
    # Initialize deduplicator
    deduplicator = SemanticImageDeduplicator(
        model_name=args.model,
        similarity_threshold=args.threshold
    )
    
    # Get image files
    image_files = get_image_files(args.input_dir)
    print(f"Found {len(image_files)} images in {args.input_dir}")
    
    # Extract or load features
    if args.cache and os.path.exists(args.cache):
        deduplicator.load_features_cache(args.cache)
    else:
        deduplicator.extract_features(image_files, save_cache=args.cache)
    
    # Find duplicates
    if args.method == 'threshold':
        duplicate_groups = deduplicator.find_duplicates_threshold()
    elif args.method == 'clustering':
        duplicate_groups = deduplicator.find_duplicates_clustering()
    elif args.method == 'faiss':
        duplicate_groups = deduplicator.find_duplicates_faiss()
    
    if len(duplicate_groups) == 0:
        print(f'#'*100)
        print("No duplicate groups found")
        print(f'#'*100)
        return
    
    print(f"Found {len(duplicate_groups)} duplicate groups")
    
    # Generate report
    deduplicator.generate_report()
    
    # Remove duplicates
    stats = deduplicator.remove_duplicates(
        output_dir=args.output_dir,
        dry_run=args.dry_run,
        selection_strategy=args.strategy,
        remove_duplicates=args.remove_duplicates
    )
    
    print("\nDeduplication Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
        


In [ ]:
model = 'resnet50'
threshold = 0.85
from platform import system
if system() == 'Linux':
    image_path = Path(r'/home/ai_easypid.work/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/main_im2_cropped_sn_images')
    cache_path = Path(r'/home/ai_easypid.wrok/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/cache')
#image_path = r'E:\data\projects\easy_end_test\ET4\2B_solder_new_model\2B_solder_new_model_unzip\main_im2_cropped_sn_images'
#cache_path = r'E:\data\projects\easy_end_test\ET4\2B_solder_new_model\2B_solder_new_model_unzip\cache'
# Initialize deduplicator
deduplicator = SemanticImageDeduplicator(
    model_name=model,
    similarity_threshold=threshold
)

In [ ]:
images_ = get_image_files(image_path)

Scanning directories: 1it [00:11, 11.24s/it]


In [ ]:
len(images_)

18359

In [ ]:
deduplicator.extract_features(images_, save_cache=cache_path)

INFO:__main__:Extracting features from 18359 images...
100%|██████████| 574/574 [45:36<00:00,  4.77s/it]


FileNotFoundError: [Errno 2] No such file or directory: '/home/ai_easypid.wrok/data/projects/easy_end_test/ET4/2B_solder_new_model/2B_solder_new_model_unzip/cache'

In [ ]:
#| export
if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--output-dir OUTPUT_DIR]
                             [--model {resnet50,efficientnet,vit}]
                             [--threshold THRESHOLD]
                             [--method {threshold,clustering,faiss}]
                             [--cache CACHE] [--dry-run]
                             [--strategy {largest_file,highest_resolution,random}]
                             input_dir
ipykernel_launcher.py: error: the following arguments are required: input_dir


SystemExit: 2

c:\Users\goni\AppData\Local\miniforge3\Lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('54_preprocessing.semantic_deduplication.ipynb')

ValueError: '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\nbs\\39_preprocessing.zero_degree_solder_pin.ipynb' is not in the subpath of '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\nbs'